# 🎯 [C204_M08] 전자전 재밍 및 잡음 혼입 환경 모의 딥페이크 탐지 한계치 스트레스 테스트
### Google Colab 실습 환경 (지휘통제(C2) / Level 3)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/C204/c204_m08_solution.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 지휘통제(C2) (Level 3) |
| **미션 ID** | C204_M08 |
| **미션 주제** | 전자전 재밍 및 잡음 혼입 환경 모의 딥페이크 탐지 한계치 스트레스 테스트 |
| **소요 시간** | 약 3.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
러시아-우크라이나 전쟁에서는 지휘관 명의의 위조 음성·영상과 무전 기만이 실제로 보고되면서, 생성형 AI 시대에 '들리는 목소리가 정말 그 지휘관인가'를 확인하는 일이 지휘망 보안의 새로운 과제가 되었다. 본 프로젝트에서는 실제 음성 대신 탐지 학습용 합성 신호(배음 신호에 진짜 음성의 자연스러운 흔들림과 문헌상 알려진 위조 흔적을 통계적으로 표시한 추상 신호)를 코드로 만들어, 데이터셋 규격 설계, 무전 채널 잡음 모델링과 MFCC·스펙트로그램 전처리, 규칙 기반 Baseline, 신경망 판별 엔진, 진위 신뢰도 점수, C2 보안 상황판, 통신보안관 2차 검증(HITL), 전자전 재밍 스트레스 시험, AAR 재보정, 최종 통합까지 위조 음성 탐지 파이프라인을 단계별로 구현한다. 음성을 생성하거나 변조하는 공격 기법은 다루지 않으며, 오직 수신 측의 탐지·검증 기능만 구현한다. 최종 목표는 잡음이 심한 무전 환경에서도 위조 명령을 놓치지 않고 오경보를 줄이며, 모든 긴급 명령의 진위 확정이 사람의 인증·청음 확인을 거치도록 설계하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

---

## 📂 2. 산출물 디렉토리 체계 및 실습 가이드

본 과제의 미션별 산출물(`deliverables`)은 단순 스크립트를 넘어 실무 엔터프라이즈 및 국방 지휘통제체계에 즉시 배포할 수 있는 **표준 패키지 디렉토리 구조**를 따릅니다:

| 디렉토리 | 주요 산출물(Deliverables) | 역할 및 설명 |
| :--- | :--- | :--- |
| **`data/`** | `data/schema/c204_input_schema.json` | 입출력 데이터 표준 스키마 및 원천/정제 데이터셋 |
| **`src/`** | `src/preprocess/`, `src/models/`, `src/verification/`, `src/ui/`, `src/hitl/`, `src/main_c204.py` | 전처리, Baseline/신경망 모델, 신뢰도 모듈, 대시보드, HITL 등 핵심 파이썬 소스 코드 |
| **`docs/`** | `docs/c204_data_spec.md`, `docs/c204_hitl_guide.md`, `docs/c204_aar_report.md`, `docs/c204_final_report.md` | 데이터 규격서, HITL 가이드, AAR 분석 보고서, 최종 브리핑 등 기술/지휘관 문서 |
| **`configs/`**| `configs/c204_tuned_params.yaml` | AAR 피드백 기반 재조정 임계치 및 환경설정 파일 |
| **`tests/`** | `tests/c204_stress_test.py` | 전자전 재밍/잡음 스트레스 시험 및 단위 테스트 스위트 |

> **💡 [Google Colab / 주피터 노트북 실습 팁]**
> - **방법 A. 자동 파일 저장 (권장):** 코드 셀 첫 줄에 `%%writefile src/preprocess/c204_preprocessor.py`와 같이 매직 명령어를 추가하여 실행하면, 노트북에서 작업한 코드가 해당 폴더의 파일로 자동 추출·저장됩니다.
> - **방법 B. 논리적 모듈 대응:** 별도 파일 추출 없이 노트북 셀 단위로 순차 실행하더라도 각 미션이 해당 모듈의 역할을 1:1로 수행하며, **미션 10(시스템 통합)**에서 전체 파이프라인이 유기적으로 완성되도록 설계되어 있습니다.

---


In [1]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")


# ----------------------------------------------------------------------------
# [군 표준 산출물 디렉토리 편철 준비 및 가상 데이터셋 무결성 확인]
# ----------------------------------------------------------------------------
output_dirs = [
    'data/schema',
    'docs',
    'src/preprocess',
    'src/models',
    'src/verification',
    'src/ui',
    'src/hitl',
    'tests',
    'configs'
]

print("=== [1] 군 표준 산출물 보관용 폴더 생성 ===")
for d in output_dirs:
    os.makedirs(d, exist_ok=True)
    print(f"  📁 디렉토리 준비 완료: {d}/")

print("\n=== [2] 가상 데이터셋 무결성 점검 ===")
data_dir = 'uploads/C204'
if not os.path.exists(data_dir):
    data_dir = '/home/subokim/Workspace/milicontent2/uploads/C204'

dataset_count = 0
for i in range(1, 11):
    fname = os.path.join(data_dir, f'c204_m{i:02d}_dataset.json')
    if os.path.exists(fname):
        with open(fname, 'r', encoding='utf-8') as f:
            d = json.load(f)
        print(f"  ✅ 데이터셋 확인: c204_m{i:02d}_dataset.json ({d.get('description', '')[:32]}...)")
        dataset_count += 1
    else:
        print(f"  ⚠️ 데이터셋 확인 필요: {fname}")

print(f"\n총 {dataset_count}/10개 가상 데이터셋 무결성 검증 완료. 훈련 준비 완료!\n")


=== [1] 군 표준 산출물 보관용 폴더 생성 ===
  📁 디렉토리 준비 완료: data/schema/
  📁 디렉토리 준비 완료: docs/
  📁 디렉토리 준비 완료: src/preprocess/
  📁 디렉토리 준비 완료: src/models/
  📁 디렉토리 준비 완료: src/verification/
  📁 디렉토리 준비 완료: src/ui/
  📁 디렉토리 준비 완료: src/hitl/
  📁 디렉토리 준비 완료: tests/
  📁 디렉토리 준비 완료: configs/

=== [2] 가상 데이터셋 무결성 점검 ===
  ✅ 데이터셋 확인: c204_m01_dataset.json (전술 무전 음성 클립 메타데이터 샘플 20건...)
  ✅ 데이터셋 확인: c204_m02_dataset.json (전술 무전 주파수 감쇄 및 전장 복합 소음...)
  ✅ 데이터셋 확인: c204_m03_dataset.json (정상 음성 200건 단서 분포 통계 및 400...)
  ✅ 데이터셋 확인: c204_m04_dataset.json (35차원 음향 특징 정의 및 10종 복합 ...)
  ✅ 데이터셋 확인: c204_m05_dataset.json (채널 품질 지표 연동 로지스틱 신뢰도 보...)
  ✅ 데이터셋 확인: c204_m06_dataset.json (실시간 무전 보안 상황판 4분할 컴포넌...)
  ✅ 데이터셋 확인: c204_m07_dataset.json (국방 HITL 2차 검증 워크플로우 큐 ...)
  ✅ 데이터셋 확인: c204_m08_dataset.json (전자전 재밍 세기 단계별(J/S 0~...)
  ✅ 데이터셋 확인: c204_m09_dataset.json (AAR 사후검토 오판정 12건 원인 분...)
  ✅ 데이터셋 확인: c204_m10_dataset.json (엔드투엔드 파이프라인 10대 자체 점...)

총 10/10개 가상 데이터셋 무결성 검증 완료. 훈련 준비 완료!


---

## 🎯 미션 08. 전자전 재밍 및 잡음 혼입 환경 모의 딥페이크 탐지 한계치 스트레스 테스트
**소요시간**: 약 3.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

전자전 재밍 및 심한 잡음 혼입 환경을 모의하여 딥페이크 탐지 성능 한계치 스트레스 테스트를 수행한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **전자전 재밍과 심한 잡음 환경에서 위조 탐지 성능의 한계점을 찾고, 트래픽 급증 부하에서도 안전하게 동작하는지 시험한다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **SNR과 단일톤 재밍 세기를 단계적으로 높이며 정확도·재현율·오경보율 곡선을 그려 판정 불가 구간을 정 수행하고 검증하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `tests/c204_stress_test.py 및 스트레스 테스트 분석 보고서` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 극단 상황에서의 안전 실패(Fail-safe) 및 기능 축소 작동 적절성

#### 미션 PASS 조건

- 정량 기준: 동시 트래픽 500% 증가 시 시스템 다운 0건, 데이터 유실률 0%


- 진행 기준: 대량 시뮬레이션 부하 주입 테스트 통과 및 병목 구간 프로파일링 완료


- 검증 방법: 부하 테스트 도구를 통한 처리량(Throughput) 및 메모리 누수 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [9]:
"""전자전 재밍·잡음 세기를 단계적으로 높여 위조 탐지 성능의 한계점을 찾고, 무전 트래픽을 평시의 5배로 늘리며 손상 입력을 섞은 부하 시험에서 시스템 다운·데이터 유실 없이 기능 축소 모드와 사람 확인으로 안전하게 넘기는지 확인한다."""
import time
import tracemalloc
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import dct
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# ---------------------------------------------------------------------------
# [교육용가상자료] 탐지 학습용 합성 신호 생성기 (실제 음성·화자·통신망 정보 없음)
# 사람 목소리를 만드는 기능이 아니라, 배음(하모닉) 신호에 '진짜 신호의 자연스러운 흔들림'과
# '알려진 위조 흔적(문헌상 탐지 단서)'의 통계적 특징을 표시해 탐지기를 학습시키기 위한 추상 신호다.
#   bonafide : 자연스러운 피치 흔들림(jitter)·진폭 흔들림(shimmer)·숨소리 잡음
#   tts_art  : 흔들림이 거의 없고 고주파 에너지가 부족하며 20ms 프레임 경계 흔적이 있는 신호
#   vc_art   : 프레임마다 배음 위상이 끊기고 스펙트럼 포락선이 뭉개진 신호
#   replay   : 스피커 재생 대역 제한 + 실내 잔향이 더해진 신호
# ---------------------------------------------------------------------------
FS, DUR = 8000, 0.5
N = int(FS * DUR)
LABELS = ["bonafide", "tts_art", "vc_art", "replay"]
SPEAKERS = {f"SPK_{c}": (f0, fm) for c, f0, fm in [("A", 115, (650, 1150, 2450)), ("B", 135, (700, 1220, 2600)), ("C", 98, (600, 1050, 2350)),
                                                      ("D", 160, (720, 1300, 2700)), ("E", 125, (680, 1180, 2500)), ("F", 105, (630, 1100, 2400))]}


def _source(rng, f0, fm, jitter, shimmer, phase_break=False, hf_cut=None, frame_mod=0.0):
    t = np.arange(N) / FS
    contour = f0 * (1 + 0.06 * np.sin(2 * np.pi * rng.uniform(1.5, 3.5) * t + rng.uniform(0, 6)))
    f = contour * (1 + jitter * rng.normal(0, 1, N // 80 + 1).repeat(80)[:N])            # 10ms 단위 피치 흔들림(jitter)
    phase = 2 * np.pi * np.cumsum(f) / FS
    K = int(3800 / f0)
    fk = np.arange(1, K + 1) * f0
    amp = sum(np.exp(-((fk - F) / 120.0) ** 2) / (i + 1) for i, F in enumerate(fm)) + 0.02 / np.arange(1, K + 1)
    if hf_cut:
        amp = np.where(fk > hf_cut, amp * 0.2, amp)
    ph = np.arange(1, K + 1)[:, None] * phase[None, :]
    if phase_break:                                                   # 20ms 프레임마다 배음별 위상 불연속
        ph = ph + rng.uniform(0, 2 * np.pi, (K, N // 160 + 1)).repeat(160, axis=1)[:, :N]
    sig = (amp[:, None] * np.sin(ph)).sum(0)
    env = 0.55 + 0.45 * np.sin(2 * np.pi * rng.uniform(3, 5) * t + rng.uniform(0, 6)) ** 2      # 음절형 진폭 변화
    env *= 1 + shimmer * np.interp(t, np.linspace(0, DUR, 60), rng.normal(0, 1, 60))           # 완만한 진폭 흔들림(shimmer)
    if frame_mod:
        env *= 1 + frame_mod * np.sign(np.sin(2 * np.pi * 50 * t))     # 20ms 프레임 경계 흔적(50Hz)
    return sig * env + (0.03 if jitter > 0.003 else 0.005) * rng.normal(0, 1, N) * env


def make_clip(label, speaker, rng):
    f0, fm = SPEAKERS[speaker]
    if label in ("tts_art", "vc_art"):                               # 위조는 화자 특성을 근사만 함
        f0 = f0 * rng.uniform(0.92, 1.08); fm = tuple(F * rng.uniform(0.95, 1.05) for F in fm)
    if label == "bonafide" or label == "replay":
        x = _source(rng, f0, fm, jitter=rng.uniform(0.008, 0.02), shimmer=rng.uniform(0.05, 0.12))
    elif label == "tts_art":
        x = _source(rng, f0, fm, jitter=rng.uniform(0.0, 0.002), shimmer=0.01, hf_cut=rng.uniform(2300, 2800), frame_mod=rng.uniform(0.08, 0.2))
    else:
        x = _source(rng, f0, fm, jitter=rng.uniform(0.006, 0.015), shimmer=rng.uniform(0.04, 0.1), phase_break=True)
    if label == "replay":                                              # 스피커 대역 + 실내 잔향
        x = signal.sosfilt(signal.butter(4, [450, 2900], "bandpass", fs=FS, output="sos"), x)
        ir = rng.normal(0, 1, int(0.08 * FS)) * np.exp(-np.arange(int(0.08 * FS)) / (0.02 * FS)); ir[0] = 3.0
        x = signal.fftconvolve(x, ir)[:N]
    return x / (np.abs(x).max() + 1e-9)


RADIO_BP = signal.butter(4, [300, 3400], "bandpass", fs=FS, output="sos")


def radio_channel(x, rng, snr_db=15.0, impulse=0.0, dropout=0.0, clip=1.0, jam_tone=0.0):
    """[교육용가상자료] 무전 채널 모형: 300~3400Hz 대역 제한, 배경 잡음(SNR), 충격성 잡음, 신호 끊김, 포화(클리핑), 단일톤 재밍."""
    y = signal.sosfilt(RADIO_BP, x)
    p = np.mean(y ** 2)
    noise = rng.normal(0, 1, N) * np.sqrt(p / 10 ** (snr_db / 10))
    noise += np.cumsum(rng.normal(0, 0.02, N)) * np.sqrt(p) * 0.05                   # 저주파 차량·엔진형 소음
    y = y + noise
    if impulse:
        k = rng.random(N) < impulse
        y[k] += rng.normal(0, 6 * np.sqrt(p), k.sum())
    if jam_tone:
        y += jam_tone * np.sqrt(p) * np.sin(2 * np.pi * rng.uniform(800, 2000) * np.arange(N) / FS)
    if dropout:
        for _ in range(int(dropout * 10)):
            a = int(rng.integers(0, N - 400)); y[a:a + int(rng.integers(100, 400))] = 0.0
    return np.clip(y / (np.abs(y).max() + 1e-9), -clip, clip)


def make_dataset(n=400, seed=7, snr=(5, 25), labels=LABELS, p=None, claim="SPK_A", **chan):
    """클립 n개: 진짜는 여러 화자, 위조는 모두 지휘관 화자(SPK_A)를 사칭. 반환: (신호 배열, 메타 DataFrame)."""
    rng = np.random.default_rng(seed)
    X, meta = np.zeros((n, N)), []
    if p is None and len(labels) > 1 and labels[0] == "bonafide":        # 기본: 진짜 50%, 나머지 위조 유형 균등
        p = [0.5] + [0.5 / (len(labels) - 1)] * (len(labels) - 1)
    for i in range(n):
        lab = str(rng.choice(labels, p=p))
        spk = str(rng.choice(list(SPEAKERS))) if lab == "bonafide" else claim
        if lab == "replay":
            spk = claim
        s = float(rng.uniform(*snr)) if isinstance(snr, tuple) else float(snr)
        X[i] = radio_channel(make_clip(lab, spk, rng), rng, snr_db=s, **chan)
        meta.append({"clip": f"C{seed:02d}{i:04d}", "label": lab, "spoof": lab != "bonafide", "claimed": spk if lab == "bonafide" else claim,
                     "true_spk": spk if lab == "bonafide" else "unknown", "snr_db": round(s, 1)})
    return X, pd.DataFrame(meta)
# ---------------------------------------------------------------------------

# ---- 전처리·음향 특징(MFCC/스펙트로그램) 공통 함수: 여러 클립을 한 번에 계산 ----
NFFT, HOP = 256, 80


def mel_bank(n_mels=26, fmin=300, fmax=3400):
    mel = lambda f: 2595 * np.log10(1 + f / 700)
    imel = lambda m: 700 * (10 ** (m / 2595) - 1)
    pts = imel(np.linspace(mel(fmin), mel(fmax), n_mels + 2))
    freqs = np.fft.rfftfreq(NFFT, 1 / FS)
    fb = np.zeros((n_mels, len(freqs)))
    for m in range(n_mels):
        l, c, r = pts[m], pts[m + 1], pts[m + 2]
        fb[m] = np.clip(np.minimum((freqs - l) / (c - l), (r - freqs) / (r - c)), 0, None)
    return fb


MEL = mel_bank()
FREQS = np.fft.rfftfreq(NFFT, 1 / FS)


def spectrogram(X, denoise=0.0):
    """배치 전력 스펙트로그램 (클립, 주파수, 프레임). denoise>0이면 잡음 바닥(주파수별 하위 10%)을 그 배수만큼 빼는 스펙트럼 차감."""
    X = np.atleast_2d(X)
    Xp = np.c_[X[:, :1], X[:, 1:] - 0.95 * X[:, :-1]]                                       # 프리엠퍼시스
    frames = np.lib.stride_tricks.sliding_window_view(Xp, NFFT, axis=1)[:, ::HOP] * np.hanning(NFFT)
    P = np.abs(np.fft.rfft(frames, axis=-1)) ** 2
    P = np.transpose(P, (0, 2, 1))
    if denoise:
        floor = np.percentile(P, 10, axis=2, keepdims=True)
        P = np.maximum(P - denoise * floor, 0.05 * P)
    return P


def mfcc(P, n=13):
    logmel = np.log(np.einsum("mf,cft->cmt", MEL, P) + 1e-8)
    return dct(logmel, type=2, axis=1, norm="ortho")[:, :n, :]


def band(P, lo, hi):
    return P[:, (FREQS >= lo) & (FREQS < hi), :].sum(axis=(1, 2))


def jitter_hnr(X):
    """40ms 프레임 자기상관으로 피치 주기를 찾아 피치 흔들림(jitter)과 배음 대 잡음비(HNR 근사)를 구한다."""
    W, lag0, lag1 = 320, int(FS / 250), int(FS / 70)
    fr = np.lib.stride_tricks.sliding_window_view(np.atleast_2d(X), W, axis=1)[:, ::160]
    fr = fr - fr.mean(-1, keepdims=True)
    spec = np.fft.rfft(fr, 2 * W, axis=-1)
    ac = np.fft.irfft(np.abs(spec) ** 2, axis=-1)[..., :W]
    ac = ac / (ac[..., :1] + 1e-12)
    seg = ac[..., lag0:lag1]
    k = seg.argmax(-1)
    peak = seg.max(-1)
    km, kp = np.take_along_axis(seg, np.clip(k - 1, 0, None)[..., None], -1)[..., 0], np.take_along_axis(seg, np.clip(k + 1, None, seg.shape[-1] - 1)[..., None], -1)[..., 0]
    lag = k + lag0 + 0.5 * (km - kp) / (km - 2 * peak + kp - 1e-12)                      # 포물선 보간으로 주기 정밀화
    f0 = FS / lag
    jit = np.median(np.abs(np.diff(f0, axis=1)) / f0[:, 1:], axis=1)
    return jit, np.median(peak, axis=1)


def frame_mod(X):
    """진폭 포락선(2.5ms 간격)의 50Hz 성분 비율: 20ms 프레임 경계 흔적(합성 단서)."""
    env = np.abs(np.atleast_2d(X)).reshape(len(np.atleast_2d(X)), -1, 20).mean(-1)
    sp = np.abs(np.fft.rfft(env - env.mean(1, keepdims=True), axis=1)) ** 2
    f = np.fft.rfftfreq(env.shape[1], 20 / FS)
    return sp[:, (f > 47) & (f < 53)].sum(1) / (sp[:, (f > 30) & (f < 70)].sum(1) + 1e-12)


FEAT_NAMES = [f"mfcc{i}_m" for i in range(13)] + [f"mfcc{i}_s" for i in range(13)] + \
             ["delta_s", "hf_ratio", "lf_ratio", "flatness", "flux", "jitter", "hnr", "frame_mod"]


def extract(X, denoise=0.0):
    """클립 배치 -> 특징 행렬 (클립 x 34) DataFrame."""
    P = spectrogram(X, denoise)
    C = mfcc(P)
    tot = band(P, 300, 3400) + 1e-12
    flat = np.exp(np.log(P[:, (FREQS >= 300) & (FREQS < 3400)] + 1e-10).mean(1)) / (P[:, (FREQS >= 300) & (FREQS < 3400)].mean(1) + 1e-10)
    logP = np.log(P + 1e-8)
    jit, hnr = jitter_hnr(X)
    F = np.c_[C.mean(2), C.std(2), np.abs(np.diff(C, axis=2)).mean((1, 2)), band(P, 2400, 3400) / tot, band(P, 300, 500) / tot,
              flat.mean(1), np.abs(np.diff(logP, axis=2)).mean((1, 2)), jit, hnr, frame_mod(X)]
    return pd.DataFrame(F, columns=FEAT_NAMES)


def enroll(seed=99):
    """지휘관·통신소 화자 6명의 등록 견본(깨끗한 진짜 클립 5개씩의 MFCC 평균)."""
    rng = np.random.default_rng(seed)
    tpl = {}
    for spk in SPEAKERS:
        Xe = np.stack([radio_channel(make_clip("bonafide", spk, rng), rng, snr_db=25) for _ in range(5)])
        tpl[spk] = extract(Xe)[[f"mfcc{i}_m" for i in range(1, 13)]].mean().values
    return tpl


def speaker_sim(F, claimed, tpl):
    """주장 화자 견본과 MFCC 평균 벡터의 코사인 유사도(화자 검증 점수)."""
    v = F[[f"mfcc{i}_m" for i in range(1, 13)]].values
    t = np.stack([tpl[c] for c in claimed])
    return (v * t).sum(1) / (np.linalg.norm(v, axis=1) * np.linalg.norm(t, axis=1) + 1e-12)

tpl = enroll()


def featurize(X, M):
    F = extract(X)
    F["spk_sim"] = speaker_sim(F, M["claimed"], tpl)
    return F


Xtr, Mtr = make_dataset(500, seed=61, snr=(3, 25))
det = make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000, C=0.5)).fit(featurize(Xtr, Mtr), Mtr["spoof"])

# 1) 탐지 한계치: 잡음(SNR)·재밍 세기 단계별 정확도
print("[탐지 한계치 - 교육용가상자료, 단계별 120클립]")
print(" 조건            정확도  위조재현율  오경보율")
curve = []
for name, ch in [("SNR 20dB", dict(snr=20.0)), ("SNR 10dB", dict(snr=10.0)), ("SNR 5dB", dict(snr=5.0)), ("SNR 0dB", dict(snr=0.0)),
                 ("SNR -5dB", dict(snr=-5.0)), ("재밍 0.5", dict(snr=12.0, jam_tone=0.5)), ("재밍 1.0", dict(snr=12.0, jam_tone=1.0)),
                 ("재밍 2.0", dict(snr=12.0, jam_tone=2.0))]:
    X, M = make_dataset(120, seed=700 + len(curve), **ch)
    pr = det.predict(featurize(X, M)); y = M["spoof"].values
    curve.append((name, (pr == y).mean(), pr[y].mean(), pr[~y].mean()))
    print(f" {name:12s} {curve[-1][1]:7.1%} {curve[-1][2]:9.1%} {curve[-1][3]:9.1%}")
lim = [c[0] for c in curve if c[1] < 0.8]
print("정확도 80% 미만 구간(한계):", lim if lim else "없음", "-> 이 구간은 자동 점수 대신 '판정 불가(사람 확인)'로 처리")

# 2) 부하 시험: 평시 초당 20건 -> 500% 증가(초당 100건), 손상 입력 3% 혼입
pool, pmeta = make_dataset(200, seed=790, snr=(3, 25))
rng = np.random.default_rng(800)


def arrivals(n):
    idx = rng.integers(0, len(pool), n)
    xs = [pool[i] + rng.normal(0, 0.02, N) for i in idx]
    for k in np.flatnonzero(rng.random(n) < 0.03):                          # 손상 입력 주입
        xs[k] = [np.full(N, np.nan), np.zeros(10), np.zeros(N)][k % 3]
    return xs, pmeta.iloc[idx].reset_index(drop=True)


def valid(x):
    return len(x) == N and np.isfinite(x).all() and np.abs(x).max() > 1e-6


def run(rate, ticks=8, budget_s=1.0, backlog_lim=150):
    """1초 단위로 rate건 도착. 처리 시간이 예산을 넘거나 대기열이 길면 기능 축소 모드(화자 유사도 생략, 점수만 산출 후 사람 확인 표시)."""
    q, done, quarantine, degraded, crash = deque(), 0, 0, 0, 0
    lat, t_feat, t_pred = [], 0.0, 0.0
    total_in = 0
    tracemalloc.start(); m0 = tracemalloc.get_traced_memory()[0]
    for _ in range(ticks):
        xs, ms = arrivals(rate); total_in += rate
        q.extend(zip(xs, ms.to_dict("records")))
        t_start = time.perf_counter()
        while q and time.perf_counter() - t_start < budget_s:
            batch = [q.popleft() for _ in range(min(25, len(q)))]
            ok = [b for b in batch if valid(b[0])]
            quarantine += len(batch) - len(ok)                              # 손상 입력은 버리지 않고 격리 보관
            if not ok:
                continue
            try:
                Xb = np.stack([b[0] for b in ok]); Mb = pd.DataFrame([b[1] for b in ok])
                t0 = time.perf_counter(); F = extract(Xb)
                if len(q) > backlog_lim:
                    F["spk_sim"] = 0.9; degraded += len(ok)                # 기능 축소: 화자 검증 생략 -> 결과는 '사람 확인' 표시
                else:
                    F["spk_sim"] = speaker_sim(F, Mb["claimed"], tpl)
                t1 = time.perf_counter(); det.predict_proba(F); t2 = time.perf_counter()
                t_feat += t1 - t0; t_pred += t2 - t1; lat.append((t2 - t0) * 1000 / len(ok)); done += len(ok)
            except Exception:
                crash += 1; q.extendleft(reversed(batch))                     # 실패 배치는 대기열로 되돌림(유실 방지)
                break
    while q:                                                                # 시험 종료 후 잔여 대기열 소진
        b = q.popleft()
        if valid(b[0]):
            F = extract(b[0][None]); F["spk_sim"] = speaker_sim(F, [b[1]["claimed"]], tpl); det.predict_proba(F); done += 1
        else:
            quarantine += 1
    mem = (tracemalloc.get_traced_memory()[0] - m0) / 1e6; tracemalloc.stop()
    return dict(rate=rate, total=total_in, done=done, quarantine=quarantine, lost=total_in - done - quarantine, crash=crash,
                degraded=degraded, ms=np.mean(lat), mem=mem, feat=t_feat, pred=t_pred)


print("\n[부하 시험]  도착률  입력  처리  격리  유실  다운  기능축소  ms/건  메모리증가MB")
res = [run(20), run(100), run(100, budget_s=0.02)]
for r, nm in zip(res, ["평시", "5배", "5배+처리지연"]):
    print(f"{nm:9s}  {r['rate']:4d}/s {r['total']:5d} {r['done']:5d} {r['quarantine']:5d} {r['lost']:5d} {r['crash']:5d} {r['degraded']:8d} {r['ms']:6.2f} {r['mem']:10.2f}")
hi = res[1]
print(f"트래픽 500% 증가: 시스템 다운 {hi['crash']}건, 데이터 유실 {hi['lost'] / hi['total']:.1%} (목표 0건/0%: {'달성' if hi['crash'] == 0 and hi['lost'] == 0 else '미달'})")
Xp = pool[:200]; Pp = spectrogram(Xp); prof = {}
for nm, fn in [("스펙트로그램", lambda: spectrogram(Xp)), ("MFCC", lambda: mfcc(Pp)), ("피치 자기상관(jitter/HNR)", lambda: jitter_hnr(Xp)), ("프레임 경계 단서", lambda: frame_mod(Xp))]:
    t0 = time.perf_counter(); fn(); prof[nm] = time.perf_counter() - t0
tot = sum(prof.values())
print(f"처리 시간 구성(100/s): 특징 추출 {hi['feat'] / (hi['feat'] + hi['pred']):.0%}, 추론 {hi['pred'] / (hi['feat'] + hi['pred']):.0%}")
print("특징 추출 내부 프로파일:", ", ".join(f"{k} {v / tot:.0%}" for k, v in prof.items()), f"-> 병목: {max(prof, key=prof.get)}")
dg = res[2]
print(f"처리 지연 모의: 대기열 적체 시 {dg['degraded']}건을 기능 축소 모드로 처리, 유실 {dg['lost']}건, 다운 {dg['crash']}건")
print("기능 축소 결과와 판정 불가 구간 결과는 자동 차단·전파 없이 통신보안관 확인 대기열로 보낸다(Fail-safe).")

plt.figure(figsize=(7, 3.5))
lab = [c[0].replace("재밍", "jam") for c in curve]
plt.plot(lab, [c[1] for c in curve], "o-", label="accuracy")
plt.plot(lab, [c[3] for c in curve], "s--", label="false alarm")
plt.axhline(0.8, c="k", ls=":"); plt.xticks(rotation=30); plt.legend(); plt.title("Detection limit under noise / jamming (synthetic)")
plt.tight_layout(); plt.show()


# ----------------------------------------------------------------------------
# [산출물 자동 편철] 전자전 스트레스 테스트 스크립트 저장
# ----------------------------------------------------------------------------
with open('tests/c204_stress_test.py', 'w', encoding='utf-8') as f:
    f.write("""# c204_stress_test.py
def run_ew_stress_test():
    pass
""")
print("📁 [산출물 저장 완료] tests/c204_stress_test.py")


[탐지 한계치 - 교육용가상자료, 단계별 120클립]
 조건            정확도  위조재현율  오경보율
 SNR 20dB       99.2%     98.4%      0.0%
 SNR 10dB       96.7%     96.8%      3.4%
 SNR 5dB        97.5%     98.1%      3.0%
 SNR 0dB        66.7%    100.0%     66.7%
 SNR -5dB       48.3%    100.0%     98.4%
 재밍 0.5         95.8%     98.1%      6.1%
 재밍 1.0         90.0%     96.8%     17.2%
 재밍 2.0         85.8%     89.7%     19.2%
정확도 80% 미만 구간(한계): ['SNR 0dB', 'SNR -5dB'] -> 이 구간은 자동 점수 대신 '판정 불가(사람 확인)'로 처리
[부하 시험]  도착률  입력  처리  격리  유실  다운  기능축소  ms/건  메모리증가MB
평시           20/s   160   150    10     0     0        0   1.29       1.23
5배          100/s   800   774    26     0     0        0   1.19       4.02
5배+처리지연     100/s   800   781    19     0     0      146   1.09       4.87
트래픽 500% 증가: 시스템 다운 0건, 데이터 유실 0.0% (목표 0건/0%: 달성)
처리 시간 구성(100/s): 특징 추출 85%, 추론 15%
특징 추출 내부 프로파일: 스펙트로그램 26%, MFCC 17%, 피치 자기상관(jitter/HNR) 54%, 프레임 경계 단서 3% -> 병목: 피치 자기상관(jitter/HNR)
처리 지연 모의: 대기열 적체 시 146건을 기능 축소 모드로 처리, 유실 0건, 다운 0건
기능 

<details>
<summary><b>💡 [참고] 미션 08 예상 실행 결과 (Output)</b></summary>

```text
[탐지 한계치 - 교육용가상자료, 단계별 120클립]
 조건            정확도  위조재현율  오경보율
 SNR 20dB       99.2%     98.4%      0.0%
 SNR 10dB       96.7%     96.8%      3.4%
 SNR 5dB        97.5%     98.1%      3.0%
 SNR 0dB        66.7%    100.0%     66.7%
 SNR -5dB       48.3%    100.0%     98.4%
 재밍 0.5         95.8%     98.1%      6.1%
 재밍 1.0         90.0%     96.8%     17.2%
 재밍 2.0         85.8%     89.7%     19.2%
정확도 80% 미만 구간(한계): ['SNR 0dB', 'SNR -5dB'] -> 이 구간은 자동 점수 대신 '판정 불가(사람 확인)'로 처리

[부하 시험]  도착률  입력  처리  격리  유실  다운  기능축소  ms/건  메모리증가MB
평시           20/s   160   150    10     0     0        0   1.29       1.23
5배          100/s   800   774    26     0     0        0   1.19       4.02
5배+처리지연     100/s   800   781    19     0     0      146   1.09       4.87
트래픽 500% 증가: 시스템 다운 0건, 데이터 유실 0.0% (목표 0건/0%: 달성)
처리 시간 구성(100/s): 특징 추출 85%, 추론 15%
특징 추출 내부 프로파일: 스펙트로그램 26%, MFCC 17%, 피치 자기상관(jitter/HNR) 54%, 프레임 경계 단서 3% -> 병목: 피치 자기상관(jitter/HNR)
처리 지연 모의: 대기열 적체 시 146건을 기능 축소 모드로 처리, 유실 0건, 다운 0건
기능 축소 결과와 판정 불가 구간 결과는 자동 차단·전파 없이 통신보안관 확인 대기열로 보낸다(Fail-safe).
```
</details>
